In [1]:
import struct
import numpy as np
import math
from numpy.random import *
import WinoTran_NCHW as NCHW

In [5]:
size = 224
chn = 64
# print(224*224*64)
# print(112*112*128)
# print(56*56*256)
# print(28*28*512)
parameter = size*size*chn*2

input1 = (np.array(rand(parameter))-0.5).astype(np.float32)
des = open("input.bin","wb")
cnt = des.write(input1)
des.close()

In [18]:
inside = 224
padding =1
chn =1
bat4Conv =1

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
K = chn
MSize = M if (M%128 == 0) else math.ceil(M/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = bat4Conv * inside * inside * chn

# readin the feature map
src = open("input.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_input = input.reshape((bat4Conv,chn,inside,inside)).astype(np.float32)

inputTran = NCHW.Wino_inputTran(sample_input,padding)


In [25]:
# readin the feature map
parameter2 = 36* MSize*KSize
src = open("../src/perfTest/M1_new2.bin","rb")
context = src.read(parameter2*4)
real_context = struct.unpack(str(parameter2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
testoutput = input.reshape((36,KSize,MSize)).astype(np.float32)

In [26]:
print(inputTran.shape)
print(testoutput.shape)
print( np.sum(np.abs(inputTran-testoutput)) )

(36, 8, 3200)
(36, 8, 3200)
0.034017075


In [ ]:
#generate random parameters
M = 1280
N = 3200
K = 128
Batch = 128
parameters1 = M * K * Batch
parameters2 = N * K * Batch

input1 = (np.array(rand(parameters1))-0.5).astype(np.float32)
des = open("input.bin","wb")
cnt = des.write(input1)
des.close()

kernel = (np.array(rand(parameters2))-0.5).astype(np.float32)
des = open("filter.bin","wb")
cnt = des.write(kernel)
des.close()

In [2]:
# Top layer of design
# control the data flow and the parameters.

# verify conv result
# chwn
inside = 22
numOfFilter = 128
padding = 1
chn = 128
bat4Conv = 1
oside = inside + 2*padding - 2


parameters1 = bat4Conv * inside * inside * chn
parameters2 = numOfFilter * 9 * chn 

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)

M = (int)(blockn * blockn * bat4Conv);
MSize = M if (M%128 == 0) else math.ceil(M/128)*128

N = numOfFilter
NSize = (int((N-1)/128)+1)*128
K = chn
KSize = (int((K-1)/8)+1)*8
kernelTran = np.zeros((36,KSize,NSize)).astype(np.float32)


# readin the feature map
src = open("input.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_input = input.reshape((bat4Conv,chn,inside,inside)).astype(np.float32)


# readin the kernel map
src = open("filter.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_kernel = input.reshape((numOfFilter,chn,3,3)).astype(np.float32)


# sample_input = np.ones((chn,inside,inside,bat4Conv)).astype(np.float32)
# sample_kernel = np.ones((chn,3,3,numOfFilter)).astype(np.float32)

DEBUG =1
if (DEBUG):
    print("inside:",inside,"  inside_beta:",inside_beta)
    print("Blockn:",blockn)
    print("M:",M,"  N:",N,"  K:",K)
    print("MSize:",MSize,"  NSize:",NSize,"  KSize:",KSize)
    print("input_Shape:",sample_input.shape,"   Kernel_Shape:", sample_kernel.shape)
    
#     print(kernelTran.shape)
    

# inputTran2 = Wino_inputTran2(sample_input)
inputTran = NCHW.Wino_inputTran(sample_input,padding)
kernelTran = NCHW.Wino_kernelTran(sample_kernel)

if (DEBUG):
    print()
    print("inputTran_Shape:",inputTran.shape)
    print("kernelTran_Shape:",kernelTran.shape)
    print()
    
gemmResult = NCHW.gemm(inputTran,kernelTran,MSize,NSize,DEBUG)

if (DEBUG):
    print()
    print("gemmResult_Shape:",gemmResult.shape)

gemmTran = NCHW.Wino_OutputTran(gemmResult,oside,bat4Conv,numOfFilter)
finalOutput = NCHW.Wino_inverseTran(gemmTran, numOfFilter, bat4Conv, blockn, oside)
print(finalOutput.shape)

inside: 22   inside_beta: 26
Blockn: 6
M: 36   N: 128   K: 128
MSize: 128   NSize: 128   KSize: 128
input_Shape: (1, 128, 22, 22)    Kernel_Shape: (128, 128, 3, 3)

inputTran_Shape: (36, 128, 128)
kernelTran_Shape: (36, 128, 128)


gemmResult_Shape: (36, 128, 128)
(1, 128, 22, 22)


In [3]:
print(inputTran.shape)

(36, 128, 128)


In [9]:
testoutput = NCHW.Conv_NCHW(sample_input, sample_kernel,padding)
print(testoutput.shape)
assert(testoutput.shape == finalOutput.shape)

err = np.sum(np.abs(testoutput - finalOutput))
print(err)

KeyboardInterrupt: 

In [9]:
nInputTran = 36*MSize*KSize
# print(MSize, KSize)

#inputTran
src = open("CuOutput2.bin","rb")
context = src.read( )
# print(nInputTran)
# print(len(context)/4)
# print(len(context)/4)
# print(nInputTran)
assert(len(context) == nInputTran*4)
real_context = struct.unpack(str(nInputTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_InputTran = input.reshape((36,KSize,MSize)).astype(np.float32)


In [10]:
print(cuda_InputTran[0,:5,:5])
print(inputTran[0,:5,:5])


[[ -3.158599     6.3091307  -14.724874    -2.5694146    2.1743302 ]
 [  7.2410793   -7.2983713  -11.325698   -13.8955555  -12.745775  ]
 [ -9.700469    -0.5545466   -0.08035505  -2.5805461   11.881523  ]
 [ -4.53967      0.87254465  11.506444    -5.7495975   -4.811021  ]
 [ -1.8297265   -2.419571     6.939871    18.717005     7.74692   ]]
[[ -3.158599     6.3091307  -14.724874    -2.5694141    2.1743305 ]
 [  7.2410803   -7.298371   -11.325697   -13.895555   -12.745775  ]
 [ -9.70047     -0.55454713  -0.08035553  -2.580546    11.881523  ]
 [ -4.5396695    0.8725447   11.506444    -5.7495975   -4.8110204 ]
 [ -1.8297265   -2.419571     6.9398713   18.717003     7.74692   ]]


In [11]:
print(inputTran.shape)
print(cuda_InputTran.shape)
err = np.sum(np.abs(inputTran - cuda_InputTran))
print(err)

(36, 128, 128)
(36, 128, 128)
0.043795973


In [18]:
nInputTran = 36*MSize*KSize
nKernelTran = 36*NSize*KSize
nGemmOutput = 36*MSize*NSize
nConvOutput = bat4Conv*oside*oside*numOfFilter

#inputTran
src = open("Module1.bin","rb")
context = src.read( )
# print(nInputTran)
# print(len(context)/4)
assert(len(context) == nInputTran*4)
real_context = struct.unpack(str(nInputTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_InputTran = input.reshape((36,KSize,MSize)).astype(np.float32)


#kernelTran
src = open("Module2.bin","rb")
context = src.read()
assert(len(context) == nKernelTran*4)
real_context = struct.unpack(str(nKernelTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_FilterTran = input.reshape((36,KSize,NSize)).astype(np.float32)


#GEMMOutput
src = open("Module3.bin","rb")
context = src.read()
# print(len(context)/4)
# print(nGemmOutput)
assert(len(context) == nGemmOutput*4)
real_context = struct.unpack(str(nGemmOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_GemmOutput = input.reshape((36,MSize,NSize)).astype(np.float32)


#GEMMOutput
src = open("Asura.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_ConvOutput = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [19]:
# for inputTran

# print(cuda_InputTran.shape)
# print(inputTran.shape)
assert(cuda_InputTran.shape == inputTran.shape)
assert(cuda_FilterTran.shape == kernelTran.shape)
assert(cuda_GemmOutput.shape == gemmResult.shape)
assert(cuda_ConvOutput.shape == finalOutput.shape)

# err0 = np.sum(np.abs(cuda_InputTran - inputTran2))
err1 = np.sum(np.abs(cuda_InputTran - inputTran))
err2 = np.sum(np.abs(cuda_FilterTran - kernelTran))
# print(err0)
print(err1,err2)

err3 = np.sum(np.abs(cuda_GemmOutput - gemmResult))
print(err3)

err4 = np.sum(np.abs(cuda_ConvOutput - finalOutput))
print(err4)
# diff = finalOutput - cuda_ConvOutput

2.2351296 0.00046742253
3.5836694
9.87331


In [27]:
nConvOutput = bat4Conv*(oside)*(oside)*numOfFilter
#GEMMOutput
src = open("ConvModule_NCHW.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_ConvModule = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [28]:
assert(cuda_ConvModule.shape == finalOutput.shape)
err = np.sum(np.abs(cuda_ConvModule - finalOutput))
print(err)

10.913142


In [29]:
print(cuda_ConvModule[0,0,:5,:5])
print(finalOutput[0,0,:5,:5])

[[ 0.53605086 -4.3838997  -5.1545353  -0.15999937 -2.5982037 ]
 [ 0.902688    2.2523973   1.9864316  -3.0306797  -1.4030911 ]
 [ 0.19874859 -2.1245184   1.78311    -0.24964428  1.908009  ]
 [ 0.5811477   7.313672    3.1764297   2.6240273   1.4857343 ]
 [-0.786922   -3.0025396   1.3739996   6.295079   -0.1813246 ]]
[[ 0.53604984 -4.383898   -5.1545362  -0.1599965  -2.5982049 ]
 [ 0.9026875   2.2523968   1.9864316  -3.0306888  -1.4030887 ]
 [ 0.19874913 -2.1245165   1.783109   -0.24963965  1.9080081 ]
 [ 0.58114743  7.313676    3.1764367   2.6240232   1.4857352 ]
 [-0.7869217  -3.002542    1.3740014   6.2950673  -0.18132457]]


In [12]:
print(cuda_ConvModule[0,0,:5,:5])
print(finalOutput[0,0,:5,:5])

[[-3.1312537  -5.193768   -0.5433121  -1.1102772   5.663509  ]
 [ 0.7761052  -2.0598507   3.238768   -2.4961014  -1.5177099 ]
 [-5.3421574  -6.095705    4.498821   -0.23359299  1.4550455 ]
 [ 0.9372153  -2.0244455  -2.125268    3.711238   -4.6974382 ]
 [-3.602247   -2.5058064  -1.1003078   0.50709033  2.2523947 ]]
[[ 0.53604984 -4.383898   -5.1545362  -0.1599965  -2.5982049 ]
 [ 0.9026875   2.2523968   1.9864316  -3.0306888  -1.4030887 ]
 [ 0.19874913 -2.1245165   1.783109   -0.24963965  1.9080081 ]
 [ 0.58114743  7.313676    3.1764367   2.6240232   1.4857352 ]
 [-0.7869217  -3.002542    1.3740014   6.2950673  -0.18132457]]


In [8]:
nConvOutput = bat4Conv*oside*oside*numOfFilter
#GEMMOutput
src = open("Cu_output.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cudnn_ConvModule = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [4]:
nConvOutput = bat4Conv*oside*oside*numOfFilter
#GEMMOutput
src = open("Cu_output2.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cudnn_ConvModule2 = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [5]:
assert(cudnn_ConvModule2.shape == finalOutput.shape)
err = np.sum(np.abs(cudnn_ConvModule2 - finalOutput))
print(err)

23.044996


In [11]:
assert(cudnn_ConvModule.shape == finalOutput.shape)
err = np.sum(np.abs(cudnn_ConvModule - finalOutput))
print(err)

19.974483


In [12]:
assert(cuda_ConvModule.shape == cudnn_ConvModule.shape)
err = np.sum(np.abs(cuda_ConvModule - cudnn_ConvModule))
print(err)

20.290037


In [9]:
import sys
print(sys.executable)

/home/elon/Desktop/Workspace/env/bin/python
